In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml

datos = fetch_openml("adult", version=2, as_frame=True)
df = datos.frame
print(df.shape)
print(df["class"].value_counts(normalize=True))
df.head()

In [ ]:
df.info()
df.describe()
df.isna().sum()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df.drop(columns=["class", "fnlwgt"])
y = (df["class"] == ">50K").astype(int)

numericas = X.select_dtypes("number").columns
categoricas = X.select_dtypes(exclude="number").columns

preparar = ColumnTransformer([
    ("num", Pipeline([("imputar", SimpleImputer(strategy="median")),
                      ("escalar", StandardScaler())]), numericas),
    ("cat", Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                      ("codificar", OneHotEncoder(handle_unknown="ignore"))]), categoricas),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

algoritmos = {
    "Regresión logística": LogisticRegression(max_iter=1000),
    "Árbol de decisión": DecisionTreeClassifier(max_depth=8, random_state=42),
    "Random forest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    "k vecinos (k-NN)": KNeighborsClassifier(n_neighbors=15),
}

resultados = []
modelos = {}
for nombre, algoritmo in algoritmos.items():
    modelo = Pipeline([("preparar", preparar), ("modelo", algoritmo)])
    inicio = time.time()
    modelo.fit(X_train, y_train)
    segundos = time.time() - inicio
    pred = modelo.predict(X_test)
    modelos[nombre] = modelo
    resultados.append({
        "algoritmo": nombre,
        "exactitud": accuracy_score(y_test, pred),
        "precisión": precision_score(y_test, pred),
        "sensibilidad": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "segundos": segundos,
    })

pd.DataFrame(resultados).sort_values("F1", ascending=False).round(3)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

mejor = modelos["Random forest"]   # cambie por el algoritmo que usted seleccione
ConfusionMatrixDisplay.from_estimator(mejor, X_test, y_test,
                                      display_labels=["<=50K", ">50K"])

In [ ]:
nombres = mejor.named_steps["preparar"].get_feature_names_out()
importancia = mejor.named_steps["modelo"].feature_importances_
pd.Series(importancia, index=nombres).sort_values(ascending=False).head(10)

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

casas = fetch_california_housing(as_frame=True)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    casas.data, casas.target, test_size=0.2, random_state=42)

for nombre, reg in {"Regresión lineal": LinearRegression(),
                    "Random forest": RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)}.items():
    reg.fit(Xc_train, yc_train)
    pred = reg.predict(Xc_test)
    print(f"{nombre}: error medio = {mean_absolute_error(yc_test, pred):.3f}  R² = {r2_score(yc_test, pred):.3f}")

In [ ]:
prueba = X_test.copy()
prueba["real"] = y_test
prueba["predicho"] = mejor.predict(X_test)
prueba.groupby("sex", observed=True)[["real", "predicho"]].mean().round(3)